<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Estudiar para el Examen: Aprendizaje Supervisado 📝
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 01
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/01%20-%20Modelos%20Lineales%20Supervisados/Para%20Dummies/00_Fundamentos_Supervisado_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del cuaderno 00 (Fundamentos del Aprendizaje Supervisado). Aprender con datos se parece muchísimo a **estudiar para un examen**: se puede estudiar bien, estudiar poco o, el error más común, **memorizar sin entender**.

Al terminar podrás explicar:
1. Qué significa que una máquina aprenda **con respuestas** (aprendizaje supervisado).
2. Por qué los datos se separan en **tres paquetes**: ejercicios, simulacro y examen final.
3. Qué es **memorizar** (sobreajuste) y qué es **estudiar muy poco** (subajuste).

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../00_Fundamentos_del_Aprendizaje_Supervisado.ipynb). Otra mirada sencilla al mismo tema: [Aprendizaje supervisado (Dummies) en Data Mining](../../../Data%20Mining/02%20-%20Clasificacion%20y%20Regresion/Para%20Dummies/00_Aprendizaje_Supervisado_Dummies.ipynb).

---
## 1. Aprender con las respuestas a la vista 📖

Imagina que preparas un examen con un **libro de ejercicios que trae las respuestas al final**. Resuelves, comparas con la respuesta correcta y vas ajustando tu forma de pensar. Eso es el **aprendizaje supervisado**: a la máquina se le dan **ejemplos con su respuesta correcta** y ella aprende la relación.

| En el colegio | En Machine Learning |
|---|---|
| El enunciado del ejercicio | Los **datos de entrada** (por ejemplo, la hora del día) |
| La respuesta del libro | La **etiqueta** (lo que queremos predecir, por ejemplo, cuántos clientes llegan) |
| Lo que aprendiste estudiando | El **modelo** |
| Un ejercicio nuevo del examen | Un **dato nuevo** que el modelo nunca vio |

Ejemplo del día a día: la **panadería de la esquina** quiere saber cuántos clientes llegan según la hora. Tenemos 60 registros de otros días (hora y número de clientes).

In [ ]:
# Datos inventados: clientes de una panaderia segun la hora del dia (con semilla fija para que siempre salgan igual)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
hora = rng.uniform(6, 20, 60)                                         # entre las 6 am y las 8 pm
clientes = 120 - 2.2 * (hora - 13) ** 2 + rng.normal(0, 8, 60)        # pico a la 1 pm, mas calmado en la madrugada y la noche

datos = pd.DataFrame({"hora": hora.round(1), "clientes": clientes.round(0)})
print("Primeros 5 registros (enunciado = hora, respuesta = clientes):")
print(datos.head().to_string(index=False))

plt.figure(figsize=(8, 4.5))
plt.scatter(hora, clientes, color="#f59e0b", edgecolor="white", s=60)
plt.xlabel("Hora del día")
plt.ylabel("Número de clientes")
plt.title("La panadería: ¿cuántos clientes llegan a cada hora?")
plt.show()

---
## 2. Tres paquetes de preguntas: ejercicios, simulacro y examen 🗂️

Un estudiante serio no usa las mismas preguntas para todo. Con los datos hacemos lo mismo y los **separamos en tres paquetes**:

| Paquete | Es como... | Para qué sirve |
|---|---|---|
| **Entrenamiento** | El libro de ejercicios | La máquina **aprende** de estos datos |
| **Validación** | El simulacro de examen | Probamos varias formas de estudiar y **elegimos la mejor** |
| **Prueba** | El examen final | Se usa **una sola vez**, al final, para saber la nota real |

> 🔒 **Regla de oro:** el examen final se guarda bajo llave. Si lo miras mientras estudias, ya no te dice nada (estarías haciendo trampa sin querer).

Separemos nuestros 60 registros. A propósito dejamos **pocos ejercicios** para que se note lo que viene después.

In [ ]:
# Separar en tres paquetes (barajamos los 60 registros y los repartimos)
orden = rng.permutation(60)
ejercicios = datos.iloc[orden[:15]]        # entrenamiento: 15 registros
simulacro = datos.iloc[orden[15:30]]       # validacion: 15 registros
examen = datos.iloc[orden[30:]]            # prueba: 30 registros

print(f"Libro de ejercicios (entrenamiento): {len(ejercicios)} registros")
print(f"Simulacro (validación):              {len(simulacro)} registros")
print(f"Examen final (prueba):               {len(examen)} registros (¡guardado bajo llave!)")

---
## 3. Estudiar poco, estudiar bien o memorizar 🧠

Pensemos en tres estudiantes que se preparan con el mismo libro de ejercicios:

| Estudiante | Qué hizo | Resultado en el examen |
|---|---|---|
| **El que estudió poco** | Solo aprendió una regla muy simple («la hora no importa mucho») | Le va mal en los ejercicios **y** en el examen → **subajuste** |
| **El que entendió** | Captó la idea general del tema | Le va bien en los ejercicios **y** en el examen → **buen ajuste** |
| **El que memorizó** | Se aprendió de memoria cada respuesta, incluidos los detalles raros | Saca 5.0 en los ejercicios, pero **se pierde** con preguntas nuevas → **sobreajuste** |

En el modelo, esto se controla con su **flexibilidad**: una línea recta es muy rígida (poco estudio) y una curva con muchísimas ondas es tan flexible que se «aprende» hasta el ruido. Midamos el **error promedio** (cuántos clientes nos equivocamos, en promedio) de tres modelos.

In [ ]:
# Tres "estudiantes": curvas de distinta flexibilidad (1 = recta rigida, 2 = curva suave, 10 = muy ondulada)
error = lambda real, pred: np.mean(np.abs(real - pred))               # error promedio: cuantos clientes nos equivocamos

filas = []
horas_dibujo = np.linspace(6, 20, 300)
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for eje, flexibilidad, nombre in zip(ejes, [1, 2, 10], ["Estudió poco", "Entendió", "Memorizó"]):
    modelo = np.polynomial.Polynomial.fit(ejercicios["hora"], ejercicios["clientes"], flexibilidad)   # aprende SOLO con los ejercicios
    filas.append({"estudiante": nombre,
                  "error en ejercicios": error(ejercicios["clientes"], modelo(ejercicios["hora"])),
                  "error en simulacro": error(simulacro["clientes"], modelo(simulacro["hora"]))})
    eje.scatter(ejercicios["hora"], ejercicios["clientes"], color="#f59e0b", edgecolor="white", s=60, label="Ejercicios")
    eje.scatter(simulacro["hora"], simulacro["clientes"], color="#6366f1", marker="x", s=45, label="Simulacro")
    eje.plot(horas_dibujo, modelo(horas_dibujo), color="#16a34a", lw=2.5)
    eje.set_ylim(-80, 140)
    eje.set_title(nombre, fontweight="bold")
    eje.set_xlabel("Hora del día")
ejes[0].set_ylabel("Número de clientes")
ejes[0].legend()
plt.tight_layout()
plt.show()

tabla = pd.DataFrame(filas).set_index("estudiante").round(1)
print(tabla)

mejor = tabla["error en simulacro"].idxmin()
print(f"\nSegún el simulacro, el mejor estudiante es: «{mejor}».")
print("Fíjate: el que memorizó tiene el error más pequeño en los ejercicios... y se equivoca más en el simulacro.")

---
## 4. ¿Existe el mejor modelo para todo? 🤔

No. Es como preguntar «¿cuál es el mejor método de estudio?»: depende de la materia. Para unos temas sirve memorizar fórmulas, para otros hay que entender ideas. En Machine Learning pasa igual: **un modelo que va muy bien en un problema puede ir mal en otro**. Por eso se prueban varios y se elige con el simulacro (validación).

Y hay un último detalle: los dos errores de estudio tienen nombre de **juego de dardos** 🎯:

| Si tus dardos... | Es como un modelo... |
|---|---|
| Caen todos juntos, pero lejos del centro | **Rígido**: siempre se equivoca de la misma manera (estudió poco) |
| Caen muy dispersos alrededor del centro | **Demasiado flexible**: cambia mucho con cada muestra (memorizó) |
| Caen juntos y en el centro | **Equilibrado**: justo lo que buscamos |

> 🔗 La versión matemática (con simulaciones) está en el [cuaderno estándar](../00_Fundamentos_del_Aprendizaje_Supervisado.ipynb), secciones de sesgo y varianza.

---
##### 🛠️ Práctica 1: Elige a tu estudiante con el simulacro y abre el examen una sola vez

Ya conoces a los tres estudiantes; ahora tú eres el profesor que decide a cuál confiarle el examen. Usa los paquetes `ejercicios`, `simulacro` y `examen`, y la función `error` de la sección anterior:

1. Haz una lista `candidatos = [1, 2, 3, 5, 10]` con las **flexibilidades** que quieres probar (1 = recta rígida, 10 = muy ondulada).
2. Para cada flexibilidad, **enseña** al modelo usando **solo** `ejercicios` (`np.polynomial.Polynomial.fit(ejercicios["hora"], ejercicios["clientes"], flex)`) y anota su error en el `simulacro` en un diccionario `errores_sim`. Guarda también cada modelo en el diccionario `modelos`.
3. Elige `mejor_flex`: la flexibilidad con el **menor error en el simulacro**.
4. Solo ahora, **abre el examen final**: calcula `error_examen` del modelo elegido y `error_examen_memorion` del modelo de flexibilidad 10.
5. Comprueba con `assert` que `error_examen < error_examen_memorion`: ¡elegir con el simulacro fue buena idea!

In [ ]:
# Escribe tu código aquí

# candidatos = [1, 2, 3, 5, 10]
# errores_sim = {}
# modelos = {}
# for flex in candidatos:
#     modelos[flex] = np.polynomial.Polynomial.fit(ejercicios["hora"], ejercicios["clientes"], ...)
#     errores_sim[flex] = error(simulacro["clientes"], modelos[flex](simulacro["hora"]))
# mejor_flex = min(errores_sim, key=errores_sim.get)       # el de menor error en el simulacro
# error_examen = error(examen["clientes"], modelos[mejor_flex](examen["hora"]))
# error_examen_memorion = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
candidatos = [1, 2, 3, 5, 10]
errores_sim, modelos = {}, {}
for flex in candidatos:
    # el modelo aprende SOLO con los ejercicios
    modelos[flex] = np.polynomial.Polynomial.fit(ejercicios["hora"], ejercicios["clientes"], flex)
    # y se evalua con el simulacro (datos que no vio)
    errores_sim[flex] = error(simulacro["clientes"], modelos[flex](simulacro["hora"]))
    print(f"Flexibilidad {flex:>2}: error en el simulacro = {errores_sim[flex]:.1f} clientes")

mejor_flex = min(errores_sim, key=errores_sim.get)                 # el de menor error en el simulacro
print(f"\nSegun el simulacro, la mejor flexibilidad es: {mejor_flex}")

# Solo ahora se abre el examen final (una unica vez)
error_examen = error(examen["clientes"], modelos[mejor_flex](examen["hora"]))
error_examen_memorion = error(examen["clientes"], modelos[10](examen["hora"]))
print(f"Examen final -> estudiante elegido: {error_examen:.1f} | el que memorizo (flex. 10): {error_examen_memorion:.1f}")
assert error_examen < error_examen_memorion
print("Elegir con el simulacro (y no con los ejercicios) nos evito confiar en el que memorizo.")
```
</details>

---


---
## Resumen en una frase 🎯

> **Aprender con respuestas es estudiar con un libro de ejercicios resueltos: se practica con un paquete, se elige con un simulacro y se evalúa una sola vez con el examen final; y cuidado con memorizar, porque saca 5.0 en los ejercicios pero se pierde en lo nuevo.**

### Lo que aprendiste hoy:
- ✅ Aprendizaje supervisado = aprender de ejemplos que traen su respuesta.
- ✅ Los datos se separan en **entrenamiento, validación y prueba**.
- ✅ **Estudiar poco** (subajuste) y **memorizar** (sobreajuste) son los dos errores clásicos.
- ✅ El error que importa se mide con datos que el modelo **no usó para aprender**.
- ✅ No hay un modelo mejor para todo: se prueban varios.

> 🎓 Continúa con [Regresión Lineal (Dummies)](01_Regresion_Lineal_Dummies.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../00_Fundamentos_del_Aprendizaje_Supervisado.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué no sirve evaluar a un modelo con los mismos datos con los que aprendió?
2. Piensa en alguien que conozcas que «se sabe todo de memoria pero no sabe aplicarlo»: ¿se parece más al modelo que estudió poco o al que memorizó? ¿Por qué?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>